# Chapter 1: Scale From Zero to Millions of Users
*System Design Interview*

## TL;DR

Scaling a system from a single server to millions of users is an **iterative process**. Each bottleneck is addressed by introducing a new architectural component:

| Stage | Solution | Benefit |
|-------|----------|---------|
| Single server | Separate web + DB | Independent scaling |
| Web tier bottleneck | **Load balancer** | Availability + horizontal scaling |
| DB read bottleneck | **Replication** (master/slave) | Read throughput + failover |
| Repeated queries | **Cache tier** (Redis/Memcached) | Reduced DB load, faster reads |
| Static asset latency | **CDN** | Geo-distributed, low-latency delivery |
| Session stickiness | **Stateless web tier** | Enables auto-scaling |
| Geographic reach | **Multi-datacenter** | Low latency worldwide |
| Tight coupling | **Message queues** | Async processing, decoupling |
| DB write/storage limit | **Sharding** | Horizontal DB scaling |

## Requirements

**Functional:**
- Serve web and mobile clients
- Handle reads and writes to a data store
- Deliver static assets efficiently

**Non-Functional:**
- High availability (no single point of failure)
- Low latency globally
- Horizontal scalability to millions of users
- Fault tolerance at every tier

## Back-of-Envelope Estimation

Suppose we are scaling to **10 million DAU**:

In [10]:
# Back-of-envelope estimation for 10M DAU system
dau = 10_000_000
avg_requests_per_user_per_day = 20  # page views + API calls

# QPS
total_requests_per_day = dau * avg_requests_per_user_per_day
qps = total_requests_per_day / (24 * 3600)
peak_qps = qps * 3  # assume 3x peak factor

print(f"DAU:              {dau:>15,}")
print(f"Requests/day:     {total_requests_per_day:>15,}")
print(f"Avg QPS:          {qps:>15,.0f}")
print(f"Peak QPS (3x):    {peak_qps:>15,.0f}")

# Cache sizing (assume 20% of requests are cache-worthy, 1 KB avg)
cache_hit_rate = 0.80
cacheable_fraction = 0.20
cached_items_per_day = total_requests_per_day * cacheable_fraction
cache_size_gb = (cached_items_per_day * 1024) / (1024**3)  # 1 KB per item
print(f"\nCacheable items/day: {cached_items_per_day:>12,.0f}")
print(f"Cache size (naive): {cache_size_gb:>12,.1f} GB")
print(f"With 80% hit rate, effective cache ~{cache_size_gb * 0.2:.1f} GB working set")

DAU:                   10,000,000
Requests/day:         200,000,000
Avg QPS:                    2,315
Peak QPS (3x):              6,944

Cacheable items/day:   40,000,000
Cache size (naive):         38.1 GB
With 80% hit rate, effective cache ~7.6 GB working set


## High-Level Design

```mermaid
graph TB
    subgraph Clients
        WEB[Web Browser]
        MOB[Mobile App]
    end

    DNS[DNS] --> WEB
    DNS --> MOB

    WEB --> CDN[CDN - Static Assets]
    MOB --> CDN

    WEB --> LB[Load Balancer]
    MOB --> LB

    subgraph Web Tier - Stateless
        LB --> WS1[Web Server 1]
        LB --> WS2[Web Server 2]
        LB --> WSN[Web Server N]
    end

    subgraph Cache Tier
        WS1 --> CACHE[(Redis / Memcached)]
        WS2 --> CACHE
        WSN --> CACHE
    end

    subgraph Data Tier
        WS1 --> MASTER[(Master DB - Writes)]
        WS2 --> MASTER
        WSN --> MASTER
        MASTER --> SLAVE1[(Slave DB 1 - Reads)]
        MASTER --> SLAVE2[(Slave DB 2 - Reads)]
    end

    subgraph Async Processing
        WS1 --> MQ[Message Queue]
        MQ --> W1[Worker 1]
        MQ --> W2[Worker N]
    end

    subgraph Shared Session Store
        WS1 -.-> SS[(NoSQL / Redis)]
        WS2 -.-> SS
        WSN -.-> SS
    end
```

## Deep Dive: Vertical vs Horizontal Scaling

| Aspect | Vertical Scaling (Scale Up) | Horizontal Scaling (Scale Out) |
|--------|---------------------------|-------------------------------|
| **Method** | Add CPU, RAM, disk to one server | Add more servers |
| **Simplicity** | Simple -- no code changes | Requires load balancer, stateless design |
| **Limit** | Hardware ceiling | Practically unlimited |
| **Failover** | Single point of failure | Built-in redundancy |
| **Cost curve** | Superlinear (expensive at top) | Linear |
| **Best for** | Small scale, quick wins | Large scale, production systems |

> **Key insight:** Vertical scaling is fine to start, but always design for horizontal scaling from the beginning.

## Deep Dive: Database Replication

```mermaid
graph LR
    APP[Application] -->|Writes| MASTER[(Master DB)]
    APP -->|Reads| SLAVE1[(Slave 1)]
    APP -->|Reads| SLAVE2[(Slave 2)]
    MASTER -->|Replication| SLAVE1
    MASTER -->|Replication| SLAVE2
```

**Failover scenarios:**
- **Slave fails:** Reads redirected to other slaves or temporarily to master
- **Master fails:** A slave is promoted to new master; data recovery scripts may be needed for missing transactions

## Deep Dive: Caching

**Read-through cache pattern:**

```mermaid
sequenceDiagram
    participant Client
    participant WebServer
    participant Cache
    participant DB

    Client->>WebServer: GET /user/123
    WebServer->>Cache: lookup user:123
    alt Cache HIT
        Cache-->>WebServer: return data
    else Cache MISS
        WebServer->>DB: SELECT * FROM users WHERE id=123
        DB-->>WebServer: user data
        WebServer->>Cache: SET user:123 with TTL
    end
    WebServer-->>Client: response
```

**Considerations:**
| Concern | Recommendation |
|---------|---------------|
| When to cache | Read-heavy, infrequently modified data |
| Expiration | Not too short (reload storm) or too long (stale data) |
| Consistency | Single-transaction updates are hard at scale |
| SPOF | Multiple cache servers across data centers |
| Eviction | LRU (most common), LFU, FIFO |

## Deep Dive: Database Sharding

```mermaid
graph TB
    APP[Application] --> HASH{hash fn: user_id % 4}
    HASH -->|0| S0[(Shard 0)]
    HASH -->|1| S1[(Shard 1)]
    HASH -->|2| S2[(Shard 2)]
    HASH -->|3| S3[(Shard 3)]
```

**Challenges:**
| Challenge | Description | Mitigation |
|-----------|-------------|------------|
| Resharding | Uneven growth exhausts shards | Consistent hashing |
| Celebrity/hotspot | One shard gets disproportionate traffic | Per-celebrity shard, further partitioning |
| Joins | Cross-shard joins are expensive | De-normalization |

## Trade-offs Summary

| Component | Benefit | Cost / Complexity |
|-----------|---------|-------------------|
| Load Balancer | Availability, horizontal scaling | Additional infrastructure, config |
| DB Replication | Read throughput, failover | Replication lag, promotion complexity |
| Cache | Reduced latency, lower DB load | Consistency, expiration tuning |
| CDN | Fast static content globally | Cost per transfer, invalidation |
| Stateless tier | Auto-scaling, simplicity | External session store needed |
| Message Queue | Decoupling, async processing | Eventual consistency, queue monitoring |
| Sharding | Horizontal DB scaling | Resharding, joins, hotspots |

## Takeaways

1. **Start simple, scale iteratively** -- single server -> separate tiers -> replicate -> cache -> CDN -> shard
2. **Eliminate single points of failure** at every tier (load balancer, replication, multi-DC)
3. **Keep the web tier stateless** to enable horizontal auto-scaling
4. **Cache aggressively** but mind consistency and expiration
5. **Use message queues** to decouple components and handle traffic spikes
6. **Shard the database** when vertical scaling hits its limits

## See Also

- [[vertical-vs-horizontal-scaling]]
- [[load-balancing]]
- [[database-replication]]
- [[caching-strategies]]
- [[cdn]]
- [[stateless-web-tier]]
- [[database-sharding]]